# Basic Adjacent Serial-Slice Alignment

Preprocess four CS13 sections and align each adjacent pair with one reproducible loop.

This curated notebook targets the current Dynamo-free Spateo API. Edit the configuration cell before execution.


## Configure the serial-slice experiment


In [ ]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np
import spateo as st
import torch

warnings.filterwarnings("ignore")
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print(f"Spateo {st.__version__}; alignment device: {DEVICE}")

SLICE_PATHS = [
    Path("/DATA/User/gaomohan/DATA/CS13_Project/cs13/add_celltype/adata_78_processed.h5ad"),
    Path("/DATA/User/gaomohan/DATA/CS13_Project/cs13/add_celltype/adata_94_processed.h5ad"),
    Path("/DATA/User/gaomohan/DATA/CS13_Project/cs13/add_celltype/adata_109_processed.h5ad"),
    Path("/DATA/User/gaomohan/DATA/CS13_Project/cs13/add_celltype/adata_118_processed.h5ad"),
]
SPATIAL_KEY = "spatial"
ANNOTATION_KEY = "celltype"
ALIGN_KEY = "align_spatial"


## Load, validate, and preprocess slices


In [ ]:
for path in SLICE_PATHS:
    if not path.is_file():
        raise FileNotFoundError(path)

slices = [st.read_h5ad(path) for path in SLICE_PATHS]
for path, adata in zip(SLICE_PATHS, slices):
    if SPATIAL_KEY not in adata.obsm or np.asarray(adata.obsm[SPATIAL_KEY]).shape[1] != 2:
        raise ValueError(f"{path.name} requires two-dimensional obsm[{SPATIAL_KEY!r}].")
    if ANNOTATION_KEY not in adata.obs:
        raise KeyError(f"{path.name} is missing obs[{ANNOTATION_KEY!r}].")
    if not adata.obs_names.is_unique:
        raise ValueError(f"{path.name} has duplicated observation identifiers.")


def preprocess_slice(adata):
    adata = st.pp.filter_cells(adata, min_expr_genes=10, inplace=False)
    adata = st.pp.filter_genes(adata, min_cells=3, inplace=False)
    if "counts_X" not in adata.layers:
        source_layer = next(
            (candidate for candidate in ("counts", "raw_counts") if candidate in adata.layers),
            None,
        )
        if source_layer is None:
            warnings.warn("No count layer found; treating X as counts. Verify this assumption.")
        adata.layers["counts_X"] = (
            adata.layers[source_layer].copy() if source_layer is not None else adata.X.copy()
        )
    st.pp.normalize_total(
        adata,
        layer="counts_X",
        out_layer="norm_X",
        target_sum=None,
        size_factor_key="Size_Factor",
        inplace=True,
    )
    st.pp.log1p_layer(
        adata,
        layer="norm_X",
        out_layer="log1p_X",
        set_X=True,
        inplace=True,
    )
    return adata


slices = [preprocess_slice(adata) for adata in slices]
st.align.group_pca(slices, pca_key="X_pca", use_hvg=False)


## Review unaligned slices


In [ ]:
st.pl.slices_2d(
    slices=slices,
    label_key=ANNOTATION_KEY,
    spatial_key=SPATIAL_KEY,
    height=4,
    center_coordinate=True,
    show_legend=True,
    legend_kwargs={"loc": "upper center", "bbox_to_anchor": (0.5, 0), "ncol": 5},
)


## Align each adjacent slice pair


In [ ]:
pairwise_alignments = {}
pairwise_mappings = {}
for left_index in range(len(slices) - 1):
    pair_name = f"slice_{left_index + 1}_to_{left_index + 2}"
    aligned_pair, mapping = st.align.morpho_align(
        models=[slices[left_index].copy(), slices[left_index + 1].copy()],
        rep_layer="X_pca",
        rep_field="obsm",
        dissimilarity="cos",
        spatial_key=SPATIAL_KEY,
        key_added=ALIGN_KEY,
        device=DEVICE,
        verbose=False,
    )
    pairwise_alignments[pair_name] = aligned_pair
    pairwise_mappings[pair_name] = mapping


## Compare rigid and non-rigid coordinates


In [ ]:
for pair_name, aligned_pair in pairwise_alignments.items():
    print(pair_name)
    st.pl.overlay_slices_2d(
        slices=aligned_pair,
        spatial_key=ALIGN_KEY,
        height=3,
        overlay_type="backward",
        show_legend=False,
    )
    st.pl.overlay_slices_2d(
        slices=aligned_pair,
        spatial_key=f"{ALIGN_KEY}_nonrigid",
        height=3,
        overlay_type="backward",
        show_legend=False,
    )
